# Analysis of Trained MeanEmbedding and AR Models with Data Assimilation

This notebook provides a comprehensive analysis and comparison between **Generic MeanEmbedding Forecast LSTM** and **Autoregressive LSTM (ARLSTM)** models, evaluating performance with and without Data Assimilation (DA).

### Key Objectives:
1. **Timing & Alignment Verification**: Run `check_discharge_timing` to verify date alignment and ensure no 1-day lookahead leak in AR input features ($y_{t-1}$).
2. **Performance Evaluation**: Compare Nash-Sutcliffe Efficiency (NSE) across basins for:
   - Generic MeanEmbedding (Simulation, No DA)
   - Generic MeanEmbedding (State DA)
   - ARLSTM (Simulation, No AR Input)
   - ARLSTM (Real AR Input $y_{t-1}$)
   - ARLSTM (Hybrid State DA + Real AR Input)
3. **Hydrograph Visualizations**: Plot observed vs predicted streamflow across basins for all model configurations.

In [1]:
# --- Imports & Discover Model Directories ---
%matplotlib inline
import os
import sys
from pathlib import Path
import glob
import torch
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

# Resolve repository root and tutorial directories for sys.path
cwd = Path(os.getcwd()).resolve()
REPO_ROOT = next((p for p in [cwd, cwd.parent, cwd.parent.parent, cwd.parent.parent.parent] if (p / 'googlehydrology').is_dir()), Path('/usr/local/google/home/kruparell/flood-forecasting'))
tut_dir = REPO_ROOT / 'tutorial' if (REPO_ROOT / 'tutorial').is_dir() else REPO_ROOT

for p in [str(REPO_ROOT), str(tut_dir), str(tut_dir / 'scripts'), str(tut_dir / 'notebooks'), str(tut_dir / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

import backend
from googlehydrology.datasetzoo.multimet import _convert_to_tensor
from googlehydrology.evaluation.tester import RegressionTester
from googlehydrology.utils.config import Config
from googlehydrology.evaluation.assimilation import Assimilation
from googlehydrology.evaluation.metrics import calculate_metrics

MODEL_RUN_DIR = tut_dir / 'model-runs'

# Locate latest Generic MeanEmbedding and ARLSTM run directories
generic_dirs = sorted(glob.glob(str(MODEL_RUN_DIR / 'generic-meanembedding-50basin_*')))
arlstm_dirs = sorted(glob.glob(str(MODEL_RUN_DIR / 'arlstm-50basin-example_*')))

if not generic_dirs:
    raise FileNotFoundError(f"No generic mean embedding run directories found in {MODEL_RUN_DIR}")
if not arlstm_dirs:
    raise FileNotFoundError(f"No ARLSTM run directories found in {MODEL_RUN_DIR}")

generic_run_dir = Path(generic_dirs[-1])
arlstm_run_dir = Path(arlstm_dirs[-1])

print(f"Generic MeanEmbedding Run Dir: {generic_run_dir}")
print(f"ARLSTM Run Dir:               {arlstm_run_dir}")

Generic MeanEmbedding Run Dir: /usr/local/google/home/kruparell/flood-forecasting/tutorial/model-runs/generic-meanembedding-50basin_2107_080323
ARLSTM Run Dir:               /usr/local/google/home/kruparell/flood-forecasting/tutorial/model-runs/arlstm-50basin-example_2107_080318


# We will now evaluate both variants: No DA vs State DA on both Generic and Autoregressive models.
# Let's verify data assimilation configuration correctness.

from googlehydrology.evaluation.tester import RegressionTester
from googlehydrology.utils.config import Config
from googlehydrology.evaluation.assimilation import Assimilation
from googlehydrology.datasetzoo.multimet import _convert_to_tensor

cfg_arlstm = Config(arlstm_run_dir / 'config.yml')
da_settings = {
    'assimilation_config': {
        'assimilation_lead_time': 0,
        'assimilation_window': 1,
        'history': 0,
        'learning_rate': 1e-4,
        'learning_rate_drop_factor': 0.5,
        'assimilation_targets': ['h_n', 'c_n'],
        'epochs': 2,
        'loss': 'MSE',
        'optimizer': 'Adam'
    }
}
cfg_arlstm.update_config(da_settings)

tester_arlstm = RegressionTester(cfg=cfg_arlstm, run_dir=arlstm_run_dir, period='test', init_model=True)
sample = tester_arlstm.dataset[0]
sample_batch = tester_arlstm.dataset.collate_fn([{k: _convert_to_tensor(k, v) for k, v in sample.items()}])

assimilation = Assimilation(cfg_arlstm.assimilation_config)
print("Running check_discharge_timing on sample dataset batch:")
diagnostics = assimilation.check_discharge_timing(sample_batch, verbose=True)

In [2]:
# --- Run Timing & AR Shift Alignment Check ---
from googlehydrology.datasetzoo.multimet import _convert_to_tensor
from googlehydrology.evaluation.tester import RegressionTester
from googlehydrology.utils.config import Config
from googlehydrology.evaluation.assimilation import Assimilation

cfg_arlstm = Config(arlstm_run_dir / 'config.yml')
da_settings = {
    'assimilation_config': {
        'assimilation_lead_time': 0,
        'assimilation_window': 1,
        'history': 10,
        'learning_rate': 1e-4,
        'learning_rate_drop_factor': 0.5,
        'assimilation_targets': ['h_n', 'c_n'],
        'epochs': 2,
        'loss': 'MSE',
        'optimizer': 'Adam'
    }
}
cfg_arlstm.update_config(da_settings)

tester_arlstm = RegressionTester(cfg=cfg_arlstm, run_dir=arlstm_run_dir, period='test', init_model=True)
sample = tester_arlstm.dataset[0]
sample_batch = tester_arlstm.dataset.collate_fn([{k: _convert_to_tensor(k, v) for k, v in sample.items()}])

assimilation = Assimilation(cfg_arlstm.assimilation_config)
print("Running check_discharge_timing on sample dataset batch:")
diagnostics = assimilation.check_discharge_timing(sample_batch, verbose=True)

Running check_discharge_timing on sample dataset batch:


In [3]:
# --- Load Evaluation Results & Calculate NSE Summary Table ---
import xarray as xr
import pandas as pd
from googlehydrology.evaluation.metrics import calculate_metrics

generic_epoch_dir = Path(sorted(glob.glob(str(generic_run_dir / 'test' / 'model_epoch*')))[-1])
arlstm_epoch_dir = Path(sorted(glob.glob(str(arlstm_run_dir / 'test' / 'model_epoch*')))[-1])

p_gen_no_da = generic_epoch_dir / 'test_results.zarr'
p_gen_da = generic_epoch_dir / 'test_results_data_assimilation.zarr'

p_ar_no_da = arlstm_epoch_dir / 'test_results.zarr'
p_ar_input = arlstm_epoch_dir / 'test_results_ar_input.zarr'
p_ar_da = arlstm_epoch_dir / 'test_results_data_assimilation.zarr'

ds_gen_no_da = xr.open_zarr(p_gen_no_da, consolidated=False).load() if p_gen_no_da.exists() else None
ds_gen_da = xr.open_zarr(p_gen_da, consolidated=False).load() if p_gen_da.exists() else None

ds_ar_no_da = xr.open_zarr(p_ar_no_da, consolidated=False).load() if p_ar_no_da.exists() else None
ds_ar_input = xr.open_zarr(p_ar_input, consolidated=False).load() if p_ar_input.exists() else None
ds_ar_da = xr.open_zarr(p_ar_da, consolidated=False).load() if p_ar_da.exists() else None

def get_1d(da):
    if da is None: return None
    da = da.squeeze().compute()
    if 'time_step' in da.dims: da = da.isel(time_step=0)
    return da

# Filter ONLY the exact common intersection of basins present in all available datasets
loaded_datasets = [ds for ds in [ds_gen_no_da, ds_gen_da, ds_ar_no_da, ds_ar_input, ds_ar_da] if ds is not None]
if not loaded_datasets:
    raise FileNotFoundError("No evaluation .zarr datasets found in model test directories!")

common_basins = set(loaded_datasets[0].basin.values)
for ds in loaded_datasets[1:]:
    common_basins = common_basins.intersection(set(ds.basin.values))

basins = sorted(list(common_basins))
print(f"Comparing models on the exact common subset of {len(basins)} basins: {basins}")

rows = []

for b in basins:
    obs = get_1d(loaded_datasets[0]['streamflow_obs'].sel(basin=b))
    if obs.isnull().all(): continue
    
    row = {'basin': b}

    if ds_gen_no_da is not None and b in ds_gen_no_da.basin.values:
        s_gen_no_da = get_1d(ds_gen_no_da['streamflow_sim'].sel(basin=b))
        row['Generic (No DA)'] = calculate_metrics(obs, s_gen_no_da, metrics=['NSE'], resolution='1D')['NSE']

    if ds_gen_da is not None and b in ds_gen_da.basin.values:
        s_gen_da = get_1d(ds_gen_da['streamflow_sim'].sel(basin=b))
        row['Generic (State DA)'] = calculate_metrics(obs, s_gen_da, metrics=['NSE'], resolution='1D')['NSE']
    
    if ds_ar_no_da is not None and b in ds_ar_no_da.basin.values:
        s_ar_no_da = get_1d(ds_ar_no_da['streamflow_sim'].sel(basin=b))
        row['ARLSTM (No DA)'] = calculate_metrics(obs, s_ar_no_da, metrics=['NSE'], resolution='1D')['NSE']

    if ds_ar_da is not None and b in ds_ar_da.basin.values:
        s_ar_da = get_1d(ds_ar_da['streamflow_sim'].sel(basin=b))
        row['ARLSTM (State DA)'] = calculate_metrics(obs, s_ar_da, metrics=['NSE'], resolution='1D')['NSE']

    rows.append(row)

df_metrics = pd.DataFrame(rows)
print("\n=== Model Evaluation Summary (NSE) ===")
print(df_metrics.to_string(index=False))

print("\n=== Summary Statistics Across Common Basins ===")
print("Median:")
print(df_metrics.median(numeric_only=True).to_string())
print("\nMean:")
print(df_metrics.mean(numeric_only=True).to_string())

df_metrics.head()

Comparing models on the exact common subset of 5 basins: ['camels_04216418', 'camels_12115000', 'camels_12377150', 'camels_12451000', 'camels_14236200']

=== Model Evaluation Summary (NSE) ===
          basin  ARLSTM (No DA)  ARLSTM (State DA)
camels_04216418        0.150593           0.151067
camels_12115000        0.165992           0.168455
camels_12377150        0.560541           0.561094
camels_12451000        0.692992           0.693669
camels_14236200        0.246055           0.245102

=== Summary Statistics Across Common Basins ===
Median:
ARLSTM (No DA)       0.246055
ARLSTM (State DA)    0.245102

Mean:
ARLSTM (No DA)       0.363235
ARLSTM (State DA)    0.363877


,basin,ARLSTM (No DA),ARLSTM (State DA)
0,camels_04216418,0.150593,0.151067
1,camels_12115000,0.165992,0.168455
2,camels_12377150,0.560541,0.561094
3,camels_12451000,0.692992,0.693669
4,camels_14236200,0.246055,0.245102


## 3. Hydrograph Visualizations

Plot observed streamflow vs predicted streamflow across different models for a selected basin.

In [4]:
# --- Hydrograph Plotting Function with Subset Selection ---
import matplotlib.pyplot as plt

def plot_hydrograph_comparison(
    basin_id,
    ds_gen_no_da=None,
    ds_gen_da=None,
    ds_ar_no_da=None,
    ds_ar_input=None,
    ds_ar_da=None,
    models_to_plot=None,
    start_date=None,
    end_date=None
):
    """Plots observed vs predicted streamflow for selected model subsets."""
    def get_basin_1d(ds, b):
        if ds is None or b not in ds.basin.values: return None
        da = ds['streamflow_sim'].sel(basin=b).squeeze().compute()
        if 'time_step' in da.dims: da = da.isel(time_step=0)
        return da

    ref_ds = ds_gen_no_da or ds_gen_da or ds_ar_no_da or ds_ar_da
    if ref_ds is None:
        raise ValueError("No dataset passed to plot_hydrograph_comparison")

    obs = ref_ds['streamflow_obs'].sel(basin=basin_id).squeeze().compute()
    if 'time_step' in obs.dims: obs = obs.isel(time_step=0)
    
    sim_gen_no_da = get_basin_1d(ds_gen_no_da, basin_id)
    sim_gen_da = get_basin_1d(ds_gen_da, basin_id)
    sim_ar_no_da = get_basin_1d(ds_ar_no_da, basin_id)
    sim_ar_input = get_basin_1d(ds_ar_input, basin_id)
    sim_ar_da = get_basin_1d(ds_ar_da, basin_id)

    dates = pd.to_datetime(obs.date.values)
    
    all_data = {
        'Date': dates,
        'Observations': obs.values,
        'Generic (No DA)': sim_gen_no_da.values if sim_gen_no_da is not None else None,
        'Generic (State DA)': sim_gen_da.values if sim_gen_da is not None else None,
    }
    if sim_ar_no_da is not None: all_data['ARLSTM (No DA)'] = sim_ar_no_da.values
    if sim_ar_input is not None: all_data['ARLSTM (No DA, Real AR Input)'] = sim_ar_input.values
    if sim_ar_da is not None: all_data['ARLSTM (State DA)'] = sim_ar_da.values

    # Filter models if models_to_plot is specified
    if models_to_plot is not None:
        plot_keys = ['Date', 'Observations'] + [m for m in models_to_plot if m in all_data]
    else:
        plot_keys = [k for k, v in all_data.items() if v is not None]

    df_plot = pd.DataFrame({k: all_data[k] for k in plot_keys if k in all_data and all_data[k] is not None}).set_index('Date')
    
    if start_date and end_date:
        df_plot = df_plot.loc[start_date:end_date]
        
    color_style_map = {
        'Observations': ('black', '-', 2.0, 0.6),
        'Generic (No DA)': ('blue', '-', 1.5, 0.8),
        'Generic (State DA)': ('red', '--', 1.5, 0.8),
        'ARLSTM (No DA)': ('purple', ':', 1.5, 0.8),
        'ARLSTM (No DA, Real AR Input)': ('orange', '-.', 1.5, 0.8),
        'ARLSTM (State DA)': ('green', '--', 1.5, 0.8),
    }

    plt.figure(figsize=(15, 6))
    for col in df_plot.columns:
        color, style, lw, alpha = color_style_map.get(col, ('grey', '-', 1.5, 0.7))
        plt.plot(df_plot.index, df_plot[col], label=col, color=color, linestyle=style, linewidth=lw, alpha=alpha)

    plt.title(f"Hydrograph Comparison for Basin: {basin_id}")
    plt.xlabel("Date")
    plt.ylabel("Streamflow (mm/day)")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

# --- Example Usage ---
sample_basin = basins[3]
plot_hydrograph_comparison(
    basin_id=sample_basin,
    ds_gen_no_da=ds_gen_no_da,
    ds_gen_da=ds_gen_da,
    ds_ar_no_da=ds_ar_no_da,
    ds_ar_input=ds_ar_input,
    ds_ar_da=ds_ar_da,
    models_to_plot=['Generic (No DA)', 'Generic (State DA)', 'ARLSTM (No DA)', 'ARLSTM (State DA)']
)

/tmp/ipykernel_2007715/1324680327.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
